# **Processing `Dataset 1`**
---

In [1]:
# %pip install datasets

In [2]:
from datasets import load_dataset

ds = load_dataset("parquet", data_files=[
    "../datasets/01 koushik7198 Samsung-samsum_processed/data/test-00000-of-00001.parquet",
    "../datasets/01 koushik7198 Samsung-samsum_processed/data/train-00000-of-00001.parquet",
    "../datasets/01 koushik7198 Samsung-samsum_processed/data/validation-00000-of-00001.parquet"
])
df_1 = ds["train"].to_pandas()
df_1.head()

,input,output
0,Mike: Have you met up with this girl Tom?\r\nT...,Tom hasn't called a girl who gave him his numb...
1,Lita: Hi Jane\r\nJane: Hi Lita\r\nLita: How's ...,Jane has a terrible headache because of her ch...
2,"Maxi: Good evening, dear Thekla! Sorry to both...",Maxi got stuck in a traffic jam on Washington ...
3,Gabrielle: <file_photo>\r\nGabrielle: <file_ph...,"Tina, Sara and Marin compliment Gabrielle on h..."
4,Belle: How is he? Are you still there?\nAndrea...,Andrea will let Belle know how the date went.


- **Getting dataset shape**

In [3]:
df_1.shape

(16131, 2)

## **Viewing Text Data**

- **Viewing raw input text**

In [4]:
import textwrap

text = repr(df_1['input'].iloc[1])
print(textwrap.fill(text, width=100))

'Lita: Hi Jane\r\nJane: Hi Lita\r\nLita: How\'s your day?\r\nJane: Oh, it\'s ok but I have a
terrible headache\r\nLita: I bet it\'s the girls\r\nJane: Sure, children are a blessing but
sometimes I\'d like to run a way\r\nLita: I know, my son is seven now but I remember when he was two
or three\r\nJane: Hahaha\r\nLita: Is Virginia still sick?\r\nJane: A little but at least she\'s not
crying all the time anymore\r\nLita: Thank God!\r\nJane: Tina was making fun of her yesterday, she
called her "farty-poop"\r\nLita: Oh, that\'s cruel!\r\nJane: I know but I must admit that after 3
days of Virginia\'s bowel sickness it made me laugh\r\nLita: Tina is about to turn 5 years old,
right?\r\nJane: Yes, next week, on Sunday'


- **Viewing textual information**

In [5]:
index = 1

text = df_1['input'].iloc[index]

for line in text.split('\n'):
    print(textwrap.fill(line, width=100))

print('-'*100)

print(textwrap.fill(df_1["output"].iloc[index], width=100))

Lita: Hi Jane
Jane: Hi Lita
Lita: How's your day?
Jane: Oh, it's ok but I have a terrible headache
Lita: I bet it's the girls
Jane: Sure, children are a blessing but sometimes I'd like to run a way
Lita: I know, my son is seven now but I remember when he was two or three
Jane: Hahaha
Lita: Is Virginia still sick?
Jane: A little but at least she's not crying all the time anymore
Lita: Thank God!
Jane: Tina was making fun of her yesterday, she called her "farty-poop"
Lita: Oh, that's cruel!
Jane: I know but I must admit that after 3 days of Virginia's bowel sickness it made me laugh
Lita: Tina is about to turn 5 years old, right?
Jane: Yes, next week, on Sunday
----------------------------------------------------------------------------------------------------
Jane has a terrible headache because of her children. Virginia is still a little sick. Tina called
her "farty-poop" yesterday. Tina's turning 5 years old next week on Sunday.


## **Text Preprocessing**

### **Normalizing line endings**

In [6]:
df_1['input_clean'] = df_1['input'].str.replace('\r\n', '\n').str.replace('\r', '\n')
df_1[['input', 'input_clean']].head()

,input,input_clean
0,Mike: Have you met up with this girl Tom?\r\nT...,Mike: Have you met up with this girl Tom?\nTom...
1,Lita: Hi Jane\r\nJane: Hi Lita\r\nLita: How's ...,Lita: Hi Jane\nJane: Hi Lita\nLita: How's your...
2,"Maxi: Good evening, dear Thekla! Sorry to both...","Maxi: Good evening, dear Thekla! Sorry to both..."
3,Gabrielle: <file_photo>\r\nGabrielle: <file_ph...,Gabrielle: <file_photo>\nGabrielle: <file_phot...
4,Belle: How is he? Are you still there?\nAndrea...,Belle: How is he? Are you still there?\nAndrea...


### **Checking dialogue structure consistency**

- **Declaring all possible dialogue structures**

In [7]:
import re

# Defining pattern for a valid "{Speaker}: {Text}" structure
pattern = re.compile(r'^[A-Za-z][A-Za-z0-9\s\.\'-]{0,40}:\s+\S.*$')

# ^                           start of the line
# [A-Za-z]                    first character must be a letter (a-z or A-Z)
# [A-Za-z0-9\s\.\'-]{0,40}    then 0 to 40 more characters, each being:
#                                - a letter (A-Za-z)             → Mike
#                                - a digit (0-9)                 → Mike123
#                                - whitespace (\s)               → Mike Dawson
#                                - a literal period (\.)         → Mr. Mike
#                                - a literal apostrophe (\')     → O'Mike
#                                - a literal hyphen (-)          → Mike-123
# :                           then a literal colon                      
# \s+                         then one or more whitespace characters (the space after the colon)
# \S                          then at least one non-whitespace character (the message can't be empty)
# .*                          then anything else, any number of times
# $                           end of the line

- **Checking with a single row's dialogue**

In [8]:
lines = [l for l in df_1['input_clean'].iloc[0].split('\n') if l.strip != '']
bad_lines = [l for l in lines if not pattern.match(l)]

print("Valid:", len(bad_lines) == 0)
print("Bad Lines:", bad_lines)

Valid: True
Bad Lines: []


- **Applying validity check throughout the entire dataset**

In [9]:
def is_valid_dialogue(text):
    lines = [l for l in text.split('\n') if l.strip != '']
    return all(pattern.match(l) for l in lines) if lines else False

df_1['is_valid'] = df_1['input_clean'].apply(is_valid_dialogue)

print(df_1['is_valid'].value_counts())

is_valid
True     16008
False      123
Name: count, dtype: int64


- **Checking out bad rows**

In [10]:
invalid_rows = df_1[~df_1['is_valid']]
invalid_rows['input_clean'].head(20)

98      Annick: have you seen mum today?\nBéatrice: i ...
603     Chris:: What are your plans now for Halloween?...
668     Jessica: hey\nJessica: the villa's party in Ib...
861     Chris :It's an appalling day today.\nMike : I ...
867      Susan: Hi girlfriend,\nJacky: Hi,\nSusan: hav...
1005    Adriana: Sorry, I guess I can't make it. I'm f...
1420    Mathéo: man, did you see the new Samsung 10? i...
1626    Matthew Spencer: Good morning\nJohnathan O’Nei...
1935    Cecilio: Notice how they're not telling how pe...
1940    Tanvi: Hey guys! Was thinking it may also be n...
2045    Ruth: Hello😍 \nAlex: Hey Ruth. How are you doi...
2148    Tom: hi, mate. doing well?\nAdam: Fine. Super ...
2250    Josh:  Man, I’m really pissed off.\nMark: What...
2901    Rayan: hey! was great seeing you at Zara the o...
2993    Beck: hey, see you at the Jinx store. we need ...
3027    Frank: where RU?\nFrank: I'm waiting in front ...
3087    Sophie: Omg Ariana Grande and Pete Davidson br...
3093    Alex: 

- **Inspecting first row**

In [11]:
def get_bad_lines(text):
    lines = [l for l in text.split('\n') if l.strip() != '']
    return [l for l in lines if not pattern.match(l)]

invalid_rows_bad_lines = invalid_rows['input_clean'].apply(get_bad_lines)
invalid_rows_bad_lines.iloc[0]  # bad lines from the first invalid row

['Béatrice: i spend the all afternoon with her.',
 "Béatrice: as usual. Asking for going back home, but she doesn't remember",
 'Béatrice: I saw this woman Angie ou Angela, and asked her about the cleaning',
 "Béatrice: she said that it's done every 2 days and the bathroom everyday.",
 "Béatrice: the bathroom wasn't very clean either. I'll buy some household products.",
 "Béatrice: ok i'll ask the caregiver to give her a shower in the morning",
 'Béatrice: elderly people are not so clean...',
 "Béatrice: Mum asks very often, but they don't have time",
 'Béatrice: Some people are nice. Véronique !',
 "Béatrice: and the room is quite confortable. Mum is fine. She's happy there"]

### **Exporting flagged dialogue rows for furthur inspection**

In [12]:
with open('../artifacts/invalid_lines_report.txt', 'w', encoding='utf-8') as f:
    for idx, bad_lines in invalid_rows_bad_lines.items():
        f.write(f"--- Row {idx} ---\n")
        for line in bad_lines:
            f.write(line + '\n')
        f.write('\n')

### **Updated cleaning rules**

1. Support names with non-ASCII letters (é, ó, ś, ł, ō, etc.) — real names include accented/Unicode characters.
2. Support digits and underscores within names — some speaker labels are bot/system names (`Tamara_reception`, `Team_V`, `Your_Health`).
3. Convert double colons ("::") to a single colon (":") — e.g. `Chris:: What...` → `Chris: What...`.
4. Add exactly one space between ":" and the message when both a name and message exist — fixes both zero-space (`Name:msg`) and multi-space (`Name:   msg`) cases.
5. Strip leading whitespace before the speaker name (e.g. " Susan: Hi..." → "Susan: Hi...").
6. Strip trailing whitespace at the end of every line.
7. Convert curly/typographic apostrophes (U+2019 "'") to straight ASCII apostrophes (U+0027 "'") — e.g. "O'Neil" → "O'Neil".
8. Trim any stray whitespace immediately around the message content itself (not just around the colon).
9. Drop lines that have no identifiable name at all — e.g. wrong/missing separator (";" , "," instead of ":"), or a "name" portion containing disallowed characters (commas, question marks, parentheses) that can't be reliably repaired.
10. Drop entirely empty dialogue rows (blank text, or all lines blank after cleaning).
11. Drop the ENTIRE datapoint (whole conversation) if any single line still fails validation after all fixes are applied — a partially-broken conversation is discarded rather than partially salvaged.

### **Cleaning the input text thoroughly**

In [13]:
# ----------------------------------------------------------------------
# Pattern definition
# ----------------------------------------------------------------------
# ^[^\W\d_]                 → first character of the name must be a Unicode LETTER
#                               (not a digit, not an underscore, not punctuation)
# [\w\s\.\'-]{0,40}         → rest of the name: Unicode word chars (letters/digits/
#                               underscore via \w), spaces, periods, apostrophes, hyphens
# :                         → literal separator colon
# \s                        → exactly one whitespace character after the colon
#                               (guaranteed by our cleaning step below)
# \S.*$                     → the message must have real (non-empty) content
DIALOGUE_LINE_PATTERN = re.compile(r"^[^\W\d_][\w\s\.\'-]{0,40}:\s\S.*$")


def clean_dialogue(text):
    """
    Cleans and validates one full dialogue (one row of df_1['input']).
    Returns the cleaned dialogue string if every line is valid,
    otherwise returns None (signalling the whole datapoint should be dropped).
    """
    # not a string (NaN, etc.) -> drop
    if not isinstance(text, str):
        return None

    # normalize all line endings to \n
    text = text.replace('\r\n', '\n').replace('\r', '\n')

    # split into lines, ignore fully blank lines
    raw_lines = [l for l in text.split('\n') if l.strip() != '']

    # empty dialogue after cleaning -> drop
    if not raw_lines:
        return None

    fixed_lines = []
    for line in raw_lines:
        # 1. strip leading/trailing whitespace around the whole line
        #    (removes leading space before name, trailing space at line end)
        line = line.strip()

        # 2. normalize curly apostrophe (U+2019) to straight ASCII apostrophe
        line = line.replace('\u2019', "'")

        # 3. collapse "::" (or more) down to a single ":"
        line = re.sub(r':{2,}', ':', line)

        # 4. fix spacing around the FIRST colon only (the name/message separator).
        #    We only touch the first colon so any colons inside the message itself
        #    (timestamps, "<file:photo>", etc.) are left untouched.
        colon_idx = line.find(':')
        if colon_idx != -1:
            name_part = line[:colon_idx]
            message_part = line[colon_idx + 1:].strip()  # trims all space around message
            if message_part:
                line = f"{name_part}: {message_part}"    # exactly one space enforced
            else:
                line = f"{name_part}:"                   # empty message -> will fail validation below

        fixed_lines.append(line)

    # final validation: every line must match the name:message pattern
    if not all(DIALOGUE_LINE_PATTERN.match(l) for l in fixed_lines):
        return None  # drop the entire datapoint if even one line is unfixable

    return '\n'.join(fixed_lines)


# ----------------------------------------------------------------------
# Apply to the whole dataset
# ----------------------------------------------------------------------
df_1['input_clean'] = df_1['input'].apply(clean_dialogue)

# keep only rows that survived cleaning (i.e. input_clean is not None)
df_1_clean = df_1[df_1['input_clean'].notna()].copy()

print(f"Original rows: {len(df_1)}")
print(f"Kept rows:     {len(df_1_clean)}")
print(f"Dropped rows:  {len(df_1) - len(df_1_clean)}")

Original rows: 16131
Kept rows:     16125
Dropped rows:  6


- **Keeping only the columns required**

In [14]:
df_1 = df_1_clean[['input_clean', 'output']]
df_1.columns = ['input', 'output']
df_1.head()

,input,output
0,Mike: Have you met up with this girl Tom?\nTom...,Tom hasn't called a girl who gave him his numb...
1,Lita: Hi Jane\nJane: Hi Lita\nLita: How's your...,Jane has a terrible headache because of her ch...
2,"Maxi: Good evening, dear Thekla! Sorry to both...",Maxi got stuck in a traffic jam on Washington ...
3,Gabrielle: <file_photo>\nGabrielle: <file_phot...,"Tina, Sara and Marin compliment Gabrielle on h..."
4,Belle: How is he? Are you still there?\nAndrea...,Andrea will let Belle know how the date went.


### **Handling `placeholders`**

In [15]:
import re

placeholder_pattern = re.compile(r'<[^<>]+>')

# <           → literal character: opening angle bracket
# [^<>]+      → one or more characters that are NOT '<' and NOT '>'
# >           → literal character: closing angle bracket

all_placeholders = df_1_clean['input_clean'].apply(lambda x: placeholder_pattern.findall(x))

- **Find unique placeholders**

In [16]:
flat_placeholders = [item for sublist in all_placeholders for item in sublist]
unique_placeholders = sorted(set(flat_placeholders))
print(unique_placeholders)

['<\nVictoria: stuDYING ->', '< link>', '< link_photo>', '<)>', "<3\nBrandie: Really ? ;p Don't worry, I'll be there soon\nEzra: I want to see you body…\nBrandie: You will ;D\nEzra: That almost sounded like a promise!!! xP\nBrandie: Maybe ;>", '<3\nClair: We need to do it again ;>', "<3\nGloria: Yea I bet ;)\nFernando: I wanna this all again and again and again…\nGloria: Wouldn't mind it at all ;P\nFernando: The moment I saw you I knew that night was going to be fun ;>", "<3\nHeidi: ok, you're right, it's awesome :O\nViolet: wait for the rest... :>", '<3\nJake: Not yet, need to stay another hour\nAlyson: Nooooo, I miss ya ;*;*\nJake: I miss you too :*\nAlyson: I cant wait until you get back home…\nJake: What do you mean?\nAlyson: We could have some fun ;>', "<3 It's a heaven on earth!\nAnna: :>", '<3 Pity though as I wanted you to join me :>', "<3 Sarah?\nSarah: ooooh i'm not sure, gotta deal with some stuff today\nSarah: i'll come as soon as possible, but can't promise you guys i'll b

In [17]:
placeholder_pattern = re.compile(r'<[^<>\n]{1,30}>')
all_placeholders = df_1_clean['input_clean'].apply(lambda x: placeholder_pattern.findall(x))
flat_placeholders = [item for sublist in all_placeholders for item in sublist]
unique_placeholders = sorted(set(flat_placeholders))

for placeholder in unique_placeholders:
    print(placeholder)

< link>
< link_photo>
<)>
<DOC>
<File:Excelsheet>
<File_line>
<File_link>
<File_photo>
<File_video>
<LOL>
<OMG>
<Stu Kim>
<`∀´>
<`ヘ´>
<`～´>
<crickets>
<emoticon>
<emoticon_:smiley:>
<emoticon_smile>
<emoticon_stuck_out_tongue>
<emoticon_thumbup>
<fIie_others>
<facepalm>
<fiile_gif>
<fil_gif>
<file _gif>
<file _other>
<file _photo>
<file _video>
<file other>
<file photo>
<file-other>
<file-photo>
<file.other>
<file:Amelia.doc>
<file:URGENT>
<file:assignment>
<file:jpg>
<file:photo>
<file:research_papers>
<file:video>
<file>
<file_ other>
<file_ photo>
<file_GIF>
<file_audio>
<file_contact>
<file_doc>
<file_docx>
<file_foto>
<file_gif>
<file_git>
<file_gps>
<file_image>
<file_link>
<file_location>
<file_movie>
<file_other>
<file_other_>
<file_others>
<file_othetr>
<file_photo >
<file_photo>
<file_photos>
<file_pic>
<file_picture>
<file_record>
<file_song>
<file_video>
<file_zip>
<flie_photo>
<fole_other>
<foto>
<gif>
<gif_file>
<kisses>
<link>
<link_other>
<link_photo>
<link_video>
<loca

- **Replacing with Canocical placeholders**

In [18]:
# ----------------------------------------------------------------------
# Canonical placeholder categories:
#   <file_photo>    - images/pictures
#   <file_video>    - videos/movies
#   <file_gif>      - gifs
#   <file_audio>    - audio/voice/songs
#   <file_doc>      - documents/spreadsheets/attachments
#   <file_location> - shared location/GPS
#   <file_contact>  - shared contact card
#   <file_link>     - shared web link
#   <file_other>    - generic/unclear file attachment (catch-all)
#   <emoticon>      - text-based emoticons, kaomoji, reaction words (not real files)
# ----------------------------------------------------------------------

# ----------------------------------------------------------------------
# Canonical placeholder groups: canonical -> list of raw variants seen in data
# ----------------------------------------------------------------------
canonical_groups = {
    '<file_photo>': [
        '<file_photo>', '<File_photo>', '<file _photo>', '<file_ photo>',
        '<file_photo >', '<file photo>', '<file-photo>', '<file:photo>',
        '<file_photos>', '<file_pic>', '<file_picture>', '<photo>',
        '<photo_file>', '<foto>', '<file_foto>', '<picutre>',
        '<\u200efile_photo>', '<flie_photo>', '<file:jpg>',
    ],
    '<file_video>': [
        '<file_video>', '<File_video>', '<file _video>', '<file:video>',
        '<video>', '<video_file>', '<file_movie>',
    ],
    '<file_gif>': [
        '<file_gif>', '<fiile_gif>', '<fil_gif>', '<file _gif>',
        '<file_GIF>', '<file_git>', '<gif>', '<gif_file>',
    ],
    '<file_audio>': [
        '<file_audio>', '<file_record>', '<file_song>',
    ],
    '<file_doc>': [
        '<file_doc>', '<file_docx>', '<file:Amelia.doc>', '<File:Excelsheet>',
        '<file:research_papers>', '<file:assignment>', '<file:URGENT>',
        '<DOC>', '<file_zip>',
    ],
    '<file_location>': [
        '<location>', '<file_gps>', '<file_location>',
    ],
    '<file_contact>': [
        '<file_contact>',
    ],
    '<file_link>': [
        '<link>', '< link>', '<link_other>', '<link_photo>',
        '< link_photo>', '<link_video>', '<File_link>', '<file_link>',
    ],
    '<file_other>': [
        '<file>', '<file _other>', '<file other>', '<file-other>',
        '<file.other>', '<file_ other>', '<file_other>', '<file_other_>',
        '<file_others>', '<file_othetr>', '<fIie_others>', '<fole_other>',
        '<File_line>', '<other>', '<other_file>', '<send_file>',
    ],
    '<emoticon>': [
        '<)>', '<LOL>', '<lol>', '<OMG>', '<crickets>', '<emoticon>',
        '<emoticon_:smiley:>', '<emoticon_smile>', '<emoticon_stuck_out_tongue>',
        '<emoticon_thumbup>', '<facepalm>', '<kisses>', '<love>', '<m(__)m>',
        '<thumb up>', '<thumbsup>', '<waves>',
        '<`∀´>', '<`ヘ´>', '<`～´>', '<丶´Д｀>',
    ],
}

# ambiguous entries — not real placeholders, leave for manual review
unmapped = ['<Stu Kim>', '<moa>']

# ----------------------------------------------------------------------
# Build the flat lookup: variant -> canonical
# ----------------------------------------------------------------------
placeholder_map = {
    variant: canonical
    for canonical, variants in canonical_groups.items()
    for variant in variants
}

def normalize_placeholders(text):
    """Replace every <...> placeholder with its canonical form (unmapped ones left as-is)."""
    return placeholder_pattern.sub(lambda m: placeholder_map.get(m.group(0), m.group(0)), text)

df_1['input'] = df_1['input'].apply(normalize_placeholders)
df_1.head()

/tmp/ipykernel_16407/3875189586.py:83: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_1['input'] = df_1['input'].apply(normalize_placeholders)


,input,output
0,Mike: Have you met up with this girl Tom?\nTom...,Tom hasn't called a girl who gave him his numb...
1,Lita: Hi Jane\nJane: Hi Lita\nLita: How's your...,Jane has a terrible headache because of her ch...
2,"Maxi: Good evening, dear Thekla! Sorry to both...",Maxi got stuck in a traffic jam on Washington ...
3,Gabrielle: <file_photo>\nGabrielle: <file_phot...,"Tina, Sara and Marin compliment Gabrielle on h..."
4,Belle: How is he? Are you still there?\nAndrea...,Andrea will let Belle know how the date went.


### **Adding `End-of-Turn (EOT)` markers**

In [19]:
# ----------------------------------------------------------------------
# Add explicit end-of-turn (EOT) markers, replacing newlines between turns
# ----------------------------------------------------------------------
EOT_TOKEN = " <EOT> "

df_1['input'] = df_1['input'].str.replace('\n', EOT_TOKEN, regex=False)
df_1.head()

/tmp/ipykernel_16407/532639647.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_1['input'] = df_1['input'].str.replace('\n', EOT_TOKEN, regex=False)


,input,output
0,Mike: Have you met up with this girl Tom? <EOT...,Tom hasn't called a girl who gave him his numb...
1,Lita: Hi Jane <EOT> Jane: Hi Lita <EOT> Lita: ...,Jane has a terrible headache because of her ch...
2,"Maxi: Good evening, dear Thekla! Sorry to both...",Maxi got stuck in a traffic jam on Washington ...
3,Gabrielle: <file_photo> <EOT> Gabrielle: <file...,"Tina, Sara and Marin compliment Gabrielle on h..."
4,Belle: How is he? Are you still there? <EOT> A...,Andrea will let Belle know how the date went.


## **Exporting dataset**

In [20]:
df_1.to_csv("../cleaned_datasets/dataset_01.csv")

# **Processing `Dataset 2`**
---

In [8]:
from datasets import load_dataset

ds = load_dataset("parquet", data_files=[
    "../datasets/02 argilla FinePersonas-Conversations-Email-Summaries/data/train-00000-of-00003.parquet",
    "../datasets/02 argilla FinePersonas-Conversations-Email-Summaries/data/train-00001-of-00003.parquet",
    "../datasets/02 argilla FinePersonas-Conversations-Email-Summaries/data/train-00002-of-00003.parquet",
])
df_2 = ds["train"].to_pandas()
df_2.head()

,conversation_id,email,maximum_brevity_summary,summary,distilabel_metadata,model_name
0,0,Subject: Virtual Field Trip Collaboration\n\nH...,Sarah proposes a virtual field trip project ex...,Sarah suggests collaborating on a virtual fiel...,{'raw_input_email_summarization_0': [{'content...,Qwen/Qwen2.5-72B-Instruct
1,0,Subject: RE: Virtual Field Trip Collaboration\...,Michael agrees to collaborate on a virtual fie...,Michael is enthusiastic about collaborating on...,{'raw_input_email_summarization_0': [{'content...,Qwen/Qwen2.5-72B-Instruct
2,0,Subject: RE: Virtual Field Trip Collaboration\...,Sarah agrees to collaborate and suggests addin...,Sarah is enthusiastic about the collaboration ...,{'raw_input_email_summarization_0': [{'content...,Qwen/Qwen2.5-72B-Instruct
3,1,Subject: Need some advice on teaching climate ...,Emily is seeking advice on teaching the human ...,Emily is reaching out for advice on teaching t...,{'raw_input_email_summarization_0': [{'content...,Qwen/Qwen2.5-72B-Instruct
4,1,Subject: RE: Need some advice on teaching clim...,Michael suggests using local examples and case...,Michael is glad to provide advice on teaching ...,{'raw_input_email_summarization_0': [{'content...,Qwen/Qwen2.5-72B-Instruct


## **Viewing Text Data**

In [12]:
df_2 = df_2[['email', 'maximum_brevity_summary', 'summary']]
df_2.head()

,email,maximum_brevity_summary,summary
0,Subject: Virtual Field Trip Collaboration\n\nH...,Sarah proposes a virtual field trip project ex...,Sarah suggests collaborating on a virtual fiel...
1,Subject: RE: Virtual Field Trip Collaboration\...,Michael agrees to collaborate on a virtual fie...,Michael is enthusiastic about collaborating on...
2,Subject: RE: Virtual Field Trip Collaboration\...,Sarah agrees to collaborate and suggests addin...,Sarah is enthusiastic about the collaboration ...
3,Subject: Need some advice on teaching climate ...,Emily is seeking advice on teaching the human ...,Emily is reaching out for advice on teaching t...
4,Subject: RE: Need some advice on teaching clim...,Michael suggests using local examples and case...,Michael is glad to provide advice on teaching ...


In [13]:
import textwrap

print(textwrap.fill(df_2['email'].iloc[0], width=100))
print('-' * 100)
print(textwrap.fill(df_2['maximum_brevity_summary'].iloc[0], width=100))
print('-' * 100)
print(textwrap.fill(df_2['summary'].iloc[0], width=100))

Subject: Virtual Field Trip Collaboration  Hey Michael,  I hope you're doing well! I've been
thinking a lot about the ideas we discussed at the conference and I think I have an idea for our
collaboration project. What if we created a virtual field trip that takes students on a journey
through different cities around the world, exploring the impacts of climate change on urban
environments?   I think this could be a great way to combine our expertise in environmental and
urban geography and create an engaging learning experience for our students. Let me know what you
think!  Best, Sarah
----------------------------------------------------------------------------------------------------
Sarah proposes a virtual field trip project exploring climate change impacts in urban environments.
----------------------------------------------------------------------------------------------------
Sarah suggests collaborating on a virtual field trip project that explores the impacts of climate
change i

## **Text Preprocessing**

In [14]:
print(textwrap.fill(df_2['email'].iloc[3547], width=100))

Subject: RE: Chivalry in Medieval Europe  Dear Emily,  Thank you for reaching out, and I'm glad you
found my post thought-provoking. The contrast between the idealized chivalric code and the harsh
realities of 20th-century warfare is indeed a fascinating topic.  In World War II, the concept of
chivalry was largely abandoned as nations engaged in total war, targeting not only enemy combatants
but also civilians and resources. The Holocaust, in particular, represented a complete disregard for
any notion of honor or mercy, as millions were systematically murdered.  I believe that the
experiences of World War II and the Holocaust exposed the myth of chivalry as little more than a
romanticized ideal that did not withstand the realities of modern warfare. I would be happy to
discuss this further and provide any insights that may be relevant to your research.  Please let me
know if you have any specific questions or if you'd like to arrange a time to chat further.  Best
wishes, Jonathan Rosen